# 05b · Corridors: predicted routes on a map

Example regions for the `corridors` main variant: predicted least-cost routes from bulk mines to
dry-bulk ports, drawn over GRIP4 highway + primary road density. A visual companion to route
check R1 in `05_corridors.ipynb` (prespec §6a, §8). Loads no lights or temperature data.

Two things to look for: routes in flat terrain split into parallel strands rather than merging
(many equal-cost paths on the 8-neighbour lattice), and dedicated rail lines are missed (Carajás →
Itaqui, Mpumalanga → Richards Bay).

In [ ]:
import sys
from pathlib import Path

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import zipfile
import numpy as np, pandas as pd, rasterio
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap
from matplotlib.lines import Line2D
from src.analysis.spec import DATA

C = f"{DATA}/prepared/corridors/misc"
RAW = f"{DATA}/raw/grip"
MINE, ROUTE, INK, MUTED = "#2a78d6", "#eb6834", "#1f1f1e", "#6b6a64"



## Data: GRIP4 land and road grids (raw 5 arc-min), corridor cells, routes, mines, ports

In [ ]:
def grid(zipname, suffix):
    z = f"{RAW}/{zipname}"
    member = [n for n in zipfile.ZipFile(z).namelist() if n.endswith(suffix)][0]
    with rasterio.open(f"/vsizip/{z}/{member}") as r:
        a = r.read(1, masked=True).astype("float32").filled(np.nan); b = r.bounds
    return a, (b.left, b.right, b.bottom, b.top)

land, ext = grid("GRIP4_density_tp1.zip", "area_land_km2.asc")
tp1, _ = grid("GRIP4_density_tp1.zip", "_dens_m_km2.asc")
tp2, _ = grid("GRIP4_density_tp2.zip", "_dens_m_km2.asc")
roads = np.where(np.nan_to_num(tp1) + np.nan_to_num(tp2) > 0, np.nan_to_num(tp1) + np.nan_to_num(tp2), np.nan)

cells = pd.read_parquet(f"{C}/cells_10km.parquet", columns=["pixel_id", "lon", "lat"]).set_index("pixel_id")
routes = pd.read_parquet(f"{C}/main/routes.parquet").join(cells, on="pixel_id").sort_values(["property_id", "dist_km"])
mines = pd.read_parquet(f"{C}/main/mines.parquet")
mines = mines[mines.routed]
ports = pd.read_parquet(f"{C}/main/ports.parquet")
ports = ports[ports.used]

## Regions (title, lon0, lon1, lat0, lat1)

In [ ]:
REGIONS = [  # (title, lon0, lon1, lat0, lat1)
    ("Pilbara iron ore, Australia", 114.0, 122.5, -24.0, -19.8),
    ("Shanxi–Hebei coal, China", 109.5, 119.5, 34.5, 41.5),
    ("Mpumalanga coal, South Africa / Mozambique", 26.5, 33.5, -29.5, -23.5),
    ("Bowen Basin coal, Australia", 145.5, 151.5, -24.5, -20.0),
    ("Carajás iron ore, Brazil", -52.0, -43.0, -7.5, -2.0),
    ("Boké bauxite, Guinea", -15.0, -11.5, 9.5, 12.0),
]

## Map

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED})
fig, axes = plt.subplots(2, 3, figsize=(15, 9.2), constrained_layout=True)
land_cmap = ListedColormap(["#ecebe6"])
for ax, (title, x0, x1, y0, y1) in zip(axes.flat, REGIONS):
    ax.set_facecolor("#ffffff")
    ax.imshow(np.where(land > 0, 1, np.nan), extent=ext, cmap=land_cmap, interpolation="nearest", zorder=0)
    ax.imshow(roads, extent=ext, cmap="Greys", norm=LogNorm(5, 500), alpha=0.75, interpolation="nearest", zorder=1)
    inbox = lambda d: d[(d.lon.between(x0, x1)) & (d.lat.between(y0, y1))]
    ids = inbox(mines).property_id
    for _, r in routes[routes.property_id.isin(ids)].groupby("property_id"):
        ax.plot(r.lon, r.lat, color=ROUTE, lw=2, alpha=0.9, solid_capstyle="round", zorder=2)
    m = inbox(mines)
    ax.scatter(m.lon, m.lat, s=34, color=MINE, edgecolor="white", linewidth=1.2, zorder=3)
    p = inbox(ports)
    ax.scatter(p.lon, p.lat, s=110, marker="*", color=INK, edgecolor="white", linewidth=1, zorder=4)
    done = []
    for _, q in p.sort_values("vessel_count_dry_bulk", ascending=False).iterrows():
        # skip ports within 12% of the panel width of one already labelled
        if len(done) == 3 or any(np.hypot(q.lon - a, q.lat - b) < 0.12 * (x1 - x0) for a, b in done):
            continue
        done.append((q.lon, q.lat))
        ax.annotate(q.port_name, (q.lon, q.lat), xytext=(5, 5), textcoords="offset points", color=INK,
                    fontsize=8.5, zorder=5, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect(1 / np.cos(np.radians((y0 + y1) / 2)))
    ax.set_title(f"{title}\n{len(m)} routed bulk mines", fontsize=10, color=INK, loc="left")
    ax.tick_params(labelsize=7.5)

handles = [Line2D([], [], color=ROUTE, lw=2, label="Predicted least-cost route (mine → port)"),
           Line2D([], [], marker="o", ls="", color=MINE, mec="white", ms=7, label="Bulk mine (active 2002–2022)"),
           Line2D([], [], marker="*", ls="", color=INK, mec="white", ms=12, label="Destination port (≥50 dry-bulk ships/yr)"),
           Line2D([], [], color="#8a8985", lw=4, label="Highways + primary roads, GRIP4 (darker = denser)")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.045))
fig.suptitle("Predicted corridor routes (main variant) against mapped major roads", fontsize=12, color=INK, x=0.01, ha="left")
out = PROJECT / "output/figures/corridors_route_examples.png"
fig.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
print(out)